# Pipeline 2 — Capilares (YOLO11s-seg + tiles 3×3)

- Dataset: `dataset_v3.1_yolo11_tiled_3x3` (fora do repo)
- Classe: só **Capilar** via `classes=[0]` (ignora `microcotiledone`)
- **Agora:** sweep F1/IoU no campeão **v4 YOLO11s** (`capilar_yolo11s_tiled_v4`), sem retreinar. Artefatos em `artifacts_v4/` (não sobrescreve v3 nem 26s).
- Calibração de área nativa (pixel isotrópico)


In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path

# Force temp/cache off C: — always use D: for this project
_TMP_D = Path(r'D:\projeto_placentas_clayton\temp_ml')
_TMP_D.mkdir(parents=True, exist_ok=True)
os.environ['TEMP'] = str(_TMP_D)
os.environ['TMP'] = str(_TMP_D)
os.environ['TMPDIR'] = str(_TMP_D)
# Must be set BEFORE importing torch. Reduces CUDA fragmentation on 4GB cards.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Ultralytics global dirs must be absolute on D: (relative paths follow CWD and can hit C:)
_ULTRA_SETTINGS = Path(os.environ.get('APPDATA', '')) / 'Ultralytics' / 'settings.json'
if _ULTRA_SETTINGS.is_file():
    _cfg = json.loads(_ULTRA_SETTINGS.read_text(encoding='utf-8'))
    _cfg['datasets_dir'] = r'D:\projeto_placentas_clayton\datasets_ultralytics'
    _cfg['weights_dir'] = r'D:\projeto_placentas_clayton\weights_ultralytics'
    _cfg['runs_dir'] = r'D:\projeto_placentas_clayton\runs_ultralytics'
    _ULTRA_SETTINGS.write_text(json.dumps(_cfg, indent=2), encoding='utf-8')
    print('Ultralytics dirs pinned to D:')
    print(' ', _cfg['datasets_dir'])
    print(' ', _cfg['weights_dir'])
    print(' ', _cfg['runs_dir'])

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

print(f'TEMP/TMP -> {_TMP_D}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

Ultralytics dirs pinned to D:
  D:\projeto_placentas_clayton\datasets_ultralytics
  D:\projeto_placentas_clayton\weights_ultralytics
  D:\projeto_placentas_clayton\runs_ultralytics
TEMP/TMP -> D:\projeto_placentas_clayton\temp_ml
Ultralytics: 8.4.155
PyTorch: 2.5.1+cu121
GPU: NVIDIA GeForce GTX 1650 SUPER


In [2]:
def discover_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() and (p / 'v2').exists():
            return p
        p = p.parent
    raise RuntimeError('Repo root not found (.git + v2)')

REPO_ROOT = discover_repo_root()

CFG = {
    'data_yaml': 'v3_capilar_yolo11s/data_capilar_tiled.yaml',
    'pretrained': 'yolo11s-seg.pt',
    'model_name': 'yolo11s-seg',
    'project': 'v3_capilar_yolo11s/runs',
    'run_name': 'capilar_yolo11s_tiled_v4',  # champion (mask mAP50 0.778, ep 35)
    'epochs': 80,
    'imgsz': 640,
    'batch': 1,
    'workers': 0,          # no extra dataloader processes (VRAM/RAM)
    'patience': 30,
    'max_det': 150,        # valid: max ~56 capilares/tile
    'classes': [0],        # Capilar only
    'iou_match': 0.5,
    # Native isotropic px (bar 50um measured as 72px on stretched-640 X-axis):
    # m = (50/72)*(640/4140) um/px  ->  AREA_FACTOR = m**2
    'area_factor': (50 / 72) ** 2 * (640 / 4140) ** 2,
    'conf_candidates': [round(x, 2) for x in np.arange(0.15, 0.71, 0.02)],
    'output_root': 'v3_capilar_yolo11s/artifacts_v4',
}

data_yaml = (REPO_ROOT / CFG['data_yaml']).resolve()
out_root = (REPO_ROOT / CFG['output_root']).resolve()
out_bench = out_root / 'benchmarks'
out_reports = out_root / 'reports'
for p in (out_root, out_bench, out_reports):
    p.mkdir(parents=True, exist_ok=True)

with data_yaml.open('r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

dataset_root = Path(data_cfg['path'])
val_images = (dataset_root / data_cfg['val']).resolve()
val_labels = (dataset_root / 'valid' / 'labels').resolve()

pretrained = Path(CFG['pretrained'])
if not pretrained.is_file():
    pretrained = (REPO_ROOT / CFG['pretrained']).resolve()
if not pretrained.is_file():
    wdir = Path(r'D:\projeto_placentas_clayton\weights_ultralytics') / CFG['pretrained']
    if wdir.is_file():
        pretrained = wdir

print('repo:', REPO_ROOT)
print('data_yaml:', data_yaml)
print('dataset:', dataset_root)
print('val_images:', val_images)
print('pretrained:', pretrained, 'exists=', pretrained.is_file())
print('run_name:', CFG['run_name'])
print('output_root:', out_root)
print('area_factor (um2/px2):', CFG['area_factor'])
print('n val tiles:', len(list(val_images.glob('*.jpg'))))

repo: D:\projeto_placentas_clayton\dev\projeto-placentas
data_yaml: D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\data_capilar_tiled.yaml
dataset: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3
val_images: D:\projeto_placentas_clayton\dataset_v3.1_yolo11_tiled_3x3\valid\images
pretrained: D:\projeto_placentas_clayton\dev\projeto-placentas\yolo11s-seg.pt exists= True
run_name: capilar_yolo11s_tiled_v4
output_root: D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\artifacts_v4
area_factor (um2/px2): 0.011524823461313616
n val tiles: 243


## 1) Checkpoint (sem treino)

**v4 campeão** — `runs/capilar_yolo11s_tiled_v4/weights/best.pt`. `DO_TRAIN = False`: não treina; só aponta o sweep/IoU viz para esse peso. Saída em `artifacts_v4/`.


In [3]:
DO_TRAIN = False  # sweep/IoU viz no best.pt do v4; True só se for treinar de novo

if DO_TRAIN:
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # YOLO26s from COCO — 11s checkpoints are a different architecture
    start_w = pretrained if pretrained.is_file() else CFG['pretrained']
    print('starting weights:', start_w)

    model = YOLO(str(start_w))
    train_results = model.train(
        data=str(data_yaml),
        epochs=CFG['epochs'],
        imgsz=CFG['imgsz'],
        batch=CFG['batch'],
        workers=CFG['workers'],
        patience=CFG['patience'],
        device=0 if torch.cuda.is_available() else 'cpu',
        project=str((REPO_ROOT / CFG['project']).resolve()),
        name=CFG['run_name'],
        exist_ok=True,
        classes=CFG['classes'],
        max_det=CFG['max_det'],
        amp=True,
        plots=False,
        retina_masks=False,
        overlap_mask=True,
        mask_ratio=4,
        lr0=0.01,
        lrf=0.01,
        degrees=90.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=0.15,
        copy_paste=0.15,
        mixup=0.0,
        close_mosaic=10,
        scale=0.3,
        hsv_s=0.7,
    )
    best_pt = Path(train_results.save_dir) / 'weights' / 'best.pt'
else:
    best_pt = (REPO_ROOT / CFG['project'] / CFG['run_name'] / 'weights' / 'best.pt').resolve()

print('best.pt:', best_pt)
assert best_pt.is_file(), f'Checkpoint não encontrado: {best_pt}'

best.pt: D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\runs\capilar_yolo11s_tiled_v4\weights\best.pt


## 2) Helpers de métrica (tile-level)

Avaliação por **tile** (igual ao split de validação). Agregação por imagem-fonte + merge de borda fica para o próximo notebook.


In [4]:
CAPILAR_CLS = 0


def parse_gt_masks(label_path: Path, img_w: int, img_h: int, cls_keep: int = CAPILAR_CLS):
    masks = []
    if not label_path.exists():
        return masks
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls = int(float(parts[0]))
        if cls != cls_keep:
            continue
        # tolerate accidental locale commas in labels
        coords = np.array([float(x.replace(',', '.')) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
        coords[:, 0] *= img_w
        coords[:, 1] *= img_h
        m = np.zeros((img_h, img_w), dtype=np.uint8)
        cv2.fillPoly(m, [coords.astype(np.int32)], 1)
        masks.append(m)
    return masks


def masks_from_result(r):
    if r.masks is None:
        return []
    # r.masks.data is already mapped to original image size when retina_masks=True
    return [(m > 0.5).astype(np.uint8) for m in r.masks.data.cpu().numpy()]


def iou(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union else 0.0


def greedy_match(pred_masks, gt_masks, thr: float):
    pairs = []
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            s = iou(pm, gm)
            if s >= thr:
                pairs.append((s, i, j))
    pairs.sort(reverse=True)
    used_p, used_g, matched = set(), set(), []
    for s, i, j in pairs:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        matched.append((s, i, j))
    return matched


def eval_at_conf(model: YOLO, conf: float):
    img_paths = sorted(val_images.glob('*.jpg'))
    tp = fp = fn = 0
    ious = []
    gt_area = 0
    pred_area = 0

    for img_path in img_paths:
        im = cv2.imread(str(img_path))
        h, w = im.shape[:2]
        lbl = val_labels / f'{img_path.stem}.txt'
        gt_masks = parse_gt_masks(lbl, w, h)

        r = model.predict(
            source=str(img_path),
            conf=conf,
            imgsz=CFG['imgsz'],
            retina_masks=True,
            max_det=CFG['max_det'],
            classes=CFG['classes'],
            verbose=False,
        )[0]
        pred_masks = masks_from_result(r)

        # ensure mask spatial size == image size
        fixed = []
        for m in pred_masks:
            if m.shape[0] != h or m.shape[1] != w:
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            fixed.append(m)
        pred_masks = fixed

        matches = greedy_match(pred_masks, gt_masks, CFG['iou_match'])
        tp += len(matches)
        fp += len(pred_masks) - len(matches)
        fn += len(gt_masks) - len(matches)
        ious.extend([s for s, _, _ in matches])

        gt_area += int(sum(int(m.sum()) for m in gt_masks))
        pred_area += int(sum(int(m.sum()) for m in pred_masks))

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    area_rel_err = abs(pred_area - gt_area) / gt_area if gt_area else 0.0
    return {
        'conf': conf,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'mean_iou': mean_iou,
        'gt_area_px': gt_area,
        'pred_area_px': pred_area,
        'gt_area_um2': gt_area * CFG['area_factor'],
        'pred_area_um2': pred_area * CFG['area_factor'],
        'area_rel_error': area_rel_err,
        'score': 0.6 * f1 + 0.3 * mean_iou + 0.1 * (1.0 - area_rel_err),
    }

print('helpers ok')

helpers ok


## 3) Confidence sweep


In [5]:
eval_model = YOLO(str(best_pt))
rows = []
t0 = time.time()
for conf in CFG['conf_candidates']:
    row = eval_at_conf(eval_model, conf)
    rows.append(row)
    print(
        f"conf={conf:.2f}  F1={row['f1']:.4f}  IoU={row['mean_iou']:.4f}  "
        f"area_err={row['area_rel_error']:.4f}  score={row['score']:.4f}"
    )

sweep_df = pd.DataFrame(rows).sort_values('score', ascending=False)
sweep_path = out_bench / 'validation_conf_sweep.csv'
sweep_df.to_csv(sweep_path, index=False)

best = sweep_df.iloc[0].to_dict()
selected = {
    'model': CFG.get('model_name', 'yolo11s-seg'),
    'run_name': CFG['run_name'],
    'checkpoint': str(best_pt),
    'best_conf': float(best['conf']),
    'f1': float(best['f1']),
    'mean_iou': float(best['mean_iou']),
    'area_rel_error': float(best['area_rel_error']),
    'weighted_score': float(best['score']),
    'area_factor_um2_per_px2': CFG['area_factor'],
    'classes': CFG['classes'],
    'notes': 'Tile-level metrics on Capilar only; field-level merge TBD',
}
(out_bench / 'selected_confidence.json').write_text(json.dumps(selected, indent=2), encoding='utf-8')

print('\n=== BEST ===')
print(json.dumps(selected, indent=2))
print(f'sweep saved: {sweep_path}')
print(f'elapsed: {time.time() - t0:.1f}s')

d:\MINICONDA3\ENVS\PROJETO_PLACENTAS\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


conf=0.15  F1=0.6886  IoU=0.7898  area_err=0.6574  score=0.6497
conf=0.17  F1=0.7018  IoU=0.7901  area_err=0.5409  score=0.6797
conf=0.19  F1=0.7131  IoU=0.7906  area_err=0.4438  score=0.7050
conf=0.21  F1=0.7204  IoU=0.7911  area_err=0.3498  score=0.7276
conf=0.23  F1=0.7282  IoU=0.7911  area_err=0.2727  score=0.7469
conf=0.25  F1=0.7324  IoU=0.7923  area_err=0.2116  score=0.7616
conf=0.27  F1=0.7323  IoU=0.7931  area_err=0.1507  score=0.7740
conf=0.29  F1=0.7281  IoU=0.7936  area_err=0.0951  score=0.7831
conf=0.31  F1=0.7288  IoU=0.7940  area_err=0.0373  score=0.7951
conf=0.33  F1=0.7258  IoU=0.7951  area_err=0.0191  score=0.7976
conf=0.35  F1=0.7213  IoU=0.7965  area_err=0.0752  score=0.7846
conf=0.37  F1=0.7178  IoU=0.7976  area_err=0.1179  score=0.7746
conf=0.39  F1=0.7093  IoU=0.7988  area_err=0.1607  score=0.7622
conf=0.41  F1=0.6996  IoU=0.8002  area_err=0.2000  score=0.7498
conf=0.43  F1=0.6897  IoU=0.8019  area_err=0.2514  score=0.7351
conf=0.45  F1=0.6761  IoU=0.8040  area_e

## 4) Relatório por tile no melhor conf


In [6]:
best_conf = float(selected['best_conf'])
totals = []
instances = []

for img_path in sorted(val_images.glob('*.jpg')):
    im = cv2.imread(str(img_path))
    h, w = im.shape[:2]
    gt_masks = parse_gt_masks(val_labels / f'{img_path.stem}.txt', w, h)
    r = eval_model.predict(
        source=str(img_path),
        conf=best_conf,
        imgsz=CFG['imgsz'],
        retina_masks=True,
        max_det=CFG['max_det'],
        classes=CFG['classes'],
        verbose=False,
    )[0]
    pred_masks = masks_from_result(r)
    pred_masks = [
        cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST) if m.shape[:2] != (h, w) else m
        for m in pred_masks
    ]
    matches = greedy_match(pred_masks, gt_masks, CFG['iou_match'])
    matched_p = {i for _, i, _ in matches}
    matched_g = {j for _, _, j in matches}

    gt_a = int(sum(int(m.sum()) for m in gt_masks))
    pr_a = int(sum(int(m.sum()) for m in pred_masks))
    totals.append({
        'Tile': img_path.name,
        'GT_Count': len(gt_masks),
        'AI_Count': len(pred_masks),
        'Matched': len(matches),
        'FP': len(pred_masks) - len(matches),
        'FN': len(gt_masks) - len(matches),
        'GT_Area_px': gt_a,
        'AI_Area_px': pr_a,
        'GT_Area_um2': gt_a * CFG['area_factor'],
        'AI_Area_um2': pr_a * CFG['area_factor'],
        'Area_Diff_pct': ((pr_a - gt_a) / gt_a * 100.0) if gt_a else 0.0,
    })

    for s, i, j in matches:
        ga = int(gt_masks[j].sum()); pa = int(pred_masks[i].sum())
        instances.append({
            'Tile': img_path.name, 'Match_Type': 'TP', 'AI_Index': i, 'GT_Index': j,
            'IoU': s, 'GT_Area_px': ga, 'AI_Area_px': pa,
            'GT_Area_um2': ga * CFG['area_factor'], 'AI_Area_um2': pa * CFG['area_factor'],
        })
    for i, pm in enumerate(pred_masks):
        if i in matched_p:
            continue
        pa = int(pm.sum())
        instances.append({
            'Tile': img_path.name, 'Match_Type': 'FP', 'AI_Index': i, 'GT_Index': -1,
            'IoU': 0.0, 'GT_Area_px': 0, 'AI_Area_px': pa,
            'GT_Area_um2': 0.0, 'AI_Area_um2': pa * CFG['area_factor'],
        })
    for j, gm in enumerate(gt_masks):
        if j in matched_g:
            continue
        ga = int(gm.sum())
        instances.append({
            'Tile': img_path.name, 'Match_Type': 'FN', 'AI_Index': -1, 'GT_Index': j,
            'IoU': 0.0, 'GT_Area_px': ga, 'AI_Area_px': 0,
            'GT_Area_um2': ga * CFG['area_factor'], 'AI_Area_um2': 0.0,
        })

totals_df = pd.DataFrame(totals)
inst_df = pd.DataFrame(instances)
totals_path = out_reports / 'capilar_tile_totals_report.csv'
inst_path = out_reports / 'capilar_tile_instance_report.csv'
totals_df.to_csv(totals_path, index=False)
inst_df.to_csv(inst_path, index=False)
print(totals_df[['GT_Count', 'AI_Count', 'Area_Diff_pct']].describe())
print('saved:', totals_path)
print('saved:', inst_path)

         GT_Count    AI_Count  Area_Diff_pct
count  243.000000  243.000000     243.000000
mean    22.913580   20.279835       2.782730
std     10.526076   10.477242     116.395973
min      0.000000    0.000000     -88.841792
25%     15.000000   12.000000     -17.512347
50%     22.000000   19.000000      -4.500461
75%     29.000000   26.500000       9.058151
max     56.000000   58.000000    1762.787371
saved: D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\artifacts_v4\reports\capilar_tile_totals_report.csv
saved: D:\projeto_placentas_clayton\dev\projeto-placentas\v3_capilar_yolo11s\artifacts_v4\reports\capilar_tile_instance_report.csv


## 5) IoU viz (amostra de tiles)

Verde = GT, vermelho = predição, amarelo = overlap. Gera no máximo `N_VIZ` tiles (243 overlays estouram disco/tempo).


In [7]:
import matplotlib.pyplot as plt

N_VIZ = 12
out_iou = out_root / 'iou_viz'
out_iou.mkdir(parents=True, exist_ok=True)

# prefer tiles with many capillaries so the overlay is informative
ranked = totals_df.sort_values('GT_Count', ascending=False)
paths = []
for name in ranked['Tile'].tolist():
    p = val_images / name
    if p.is_file():
        paths.append(p)
    if len(paths) >= N_VIZ:
        break

best_conf = float(selected['best_conf'])
print(f'IoU viz: {len(paths)} tiles @ conf={best_conf}')

for img_path in paths:
    im_bgr = cv2.imread(str(img_path))
    h, w = im_bgr.shape[:2]
    img = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
    gt_masks = parse_gt_masks(val_labels / f'{img_path.stem}.txt', w, h)
    r = eval_model.predict(
        source=str(img_path),
        conf=best_conf,
        imgsz=CFG['imgsz'],
        retina_masks=True,
        max_det=CFG['max_det'],
        classes=CFG['classes'],
        verbose=False,
    )[0]
    pred_masks = masks_from_result(r)
    pred_masks = [
        cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST) if m.shape[:2] != (h, w) else m
        for m in pred_masks
    ]

    gt_c = np.zeros((h, w), dtype=np.uint8)
    ai_c = np.zeros((h, w), dtype=np.uint8)
    for m in gt_masks:
        gt_c = np.logical_or(gt_c, m).astype(np.uint8)
    for m in pred_masks:
        ai_c = np.logical_or(ai_c, m).astype(np.uint8)

    inter = np.logical_and(gt_c, ai_c).sum()
    union = np.logical_or(gt_c, ai_c).sum()
    img_iou = (inter / union) if union else 0.0

    overlay = img.copy()
    overlay[gt_c == 1] = (overlay[gt_c == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
    overlay[ai_c == 1] = (overlay[ai_c == 1] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
    both = np.logical_and(gt_c, ai_c)
    overlay[both] = (overlay[both] * 0.5 + np.array([255, 255, 0]) * 0.5).astype(np.uint8)

    diff = np.zeros((h, w, 3), dtype=np.uint8)
    diff[np.logical_and(gt_c == 1, ai_c == 0)] = [0, 255, 0]
    diff[np.logical_and(gt_c == 0, ai_c == 1)] = [255, 0, 0]
    diff[both] = [255, 255, 0]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f'{img_path.name} | tile IoU={img_iou:.3f} | GT={len(gt_masks)} AI={len(pred_masks)}', fontsize=11)
    axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(overlay); axes[1].set_title('Overlay GT/AI'); axes[1].axis('off')
    axes[2].imshow(diff); axes[2].set_title('Diff'); axes[2].axis('off')
    save_path = out_iou / f'iou_viz_{img_path.stem}.png'
    fig.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print('saved', save_path.name, f'IoU={img_iou:.3f}')

print('dir:', out_iou)


IoU viz: 12 tiles @ conf=0.33
saved iou_viz_TORDILHA-G_012_jpg.rf.5a71e85cb1442ba538dc7acb7149a07d_r1c1.png IoU=0.793
saved iou_viz_TORDILHA-D_011_jpg.rf.67cbd65b708b681788cae0003256ca67_r2c2.png IoU=0.639
saved iou_viz_ZAINA-679-D_007_jpg.rf.34de0be2ff800849eeb730bc4890f94f_r2c1.png IoU=0.286
saved iou_viz_TP-D_009_jpg.rf.e8cdb11635df3be6c85d743284d8ecfb_r1c2.png IoU=0.676
saved iou_viz_TORDILHA-B_006_jpg.rf.235faef65a5565236d3b0050d4713580_r0c1.png IoU=0.814
saved iou_viz_ROSILHA-M-D_006_jpg.rf.4bf38cf5314617b13c7c11445fe40b04_r0c2.png IoU=0.714
saved iou_viz_TORDILHA-D_003_jpg.rf.e67cd4ad20c15ed213012bb43fa0dc3f_r2c2.png IoU=0.512
saved iou_viz_TP-D_009_jpg.rf.e8cdb11635df3be6c85d743284d8ecfb_r2c0.png IoU=0.613
saved iou_viz_TORDILHA-D_011_jpg.rf.67cbd65b708b681788cae0003256ca67_r1c0.png IoU=0.650
saved iou_viz_ROSILHA-M-D_011_jpg.rf.3a2fb2d36de8bcc90049d66dd562a41c_r0c1.png IoU=0.794
saved iou_viz_TORDILHA-B_006_jpg.rf.235faef65a5565236d3b0050d4713580_r2c2.png IoU=0.768
saved iou_v